In [0]:
import pyspark.sql.functions as F

In [0]:
spark.sql("USE CATALOG mvp_saude")
spark.sql("USE SCHEMA bronze")

DataFrame[]

In [0]:
df_staging = spark.table("mvp_saude.staging.sih_raw_staging")

In [0]:
df_bronze = df_staging \
    .withColumn("dh_ingestao_bronze", F.current_timestamp()) \
    .withColumn("fonte_origem", F.lit("DATASUS_SIH_KAGGLE_RS"))

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp_saude.bronze.sih_raw")

# Verificar o estado importado na Bronze

In [0]:
import pyspark.sql.functions as F

In [0]:
df_check = spark.table("mvp_saude.bronze.sih_raw")

In [0]:
for col_name in df_check.columns:
    df_check = df_check.withColumnRenamed(col_name, col_name.upper())

In [0]:
if "UF" in df_check.columns:
    df_check.groupBy("UF").count().orderBy(F.col("count").desc()).show(10)
elif "UF_ZI" in df_check.columns:
    df_check.groupBy("UF_ZI").count().orderBy(F.col("count").desc()).show(10)
elif "MUNIC_RES" in df_check.columns:
    # Mostra os 2 primeiros dígitos do código do município (43 = RS)
    df_check.withColumn("COD_UF", F.col("MUNIC_RES").cast("string").substr(1, 2)) \
            .groupBy("COD_UF").count().orderBy(F.col("count").desc()).show(10)

+------+-----+
| UF_ZI|count|
+------+-----+
|430000|27614|
|431490|14404|
|430460| 2662|
|430510| 2339|
|431440| 2035|
|431340| 1185|
|431680| 1108|
|431870|  910|
|431140|  855|
|430920|  647|
+------+-----+
only showing top 10 rows


In [0]:
%sql
SELECT count(*) AS total_linhas 
FROM mvp_saude.bronze.sih_raw;

total_linhas
63922


In [0]:
%sql
SELECT * 
FROM mvp_saude.bronze.sih_raw 
LIMIT 10;

UF_ZI,ANO_CMPT,MES_CMPT,ESPEC,CGC_HOSP,N_AIH,IDENT,CEP,MUNIC_RES,NASC,SEXO,UTI_MES_IN,UTI_MES_AN,UTI_MES_AL,UTI_MES_TO,MARCA_UTI,UTI_INT_IN,UTI_INT_AN,UTI_INT_AL,UTI_INT_TO,DIAR_ACOM,QT_DIARIAS,PROC_SOLIC,PROC_REA,VAL_SH,VAL_SP,VAL_SADT,VAL_RN,VAL_ACOMP,VAL_ORTP,VAL_SANGUE,VAL_SADTSR,VAL_TRANSP,VAL_OBSANG,VAL_PED1AC,VAL_TOT,VAL_UTI,US_TOT,DT_INTER,DT_SAIDA,DIAG_PRINC,DIAG_SECUN,COBRANCA,NATUREZA,NAT_JUR,GESTAO,RUBRICA,IND_VDRL,MUNIC_MOV,COD_IDADE,IDADE,DIAS_PERM,MORTE,NACIONAL,NUM_PROC,CAR_INT,TOT_PT_SP,CPF_AUT,HOMONIMO,NUM_FILHOS,INSTRU,CID_NOTIF,CONTRACEP1,CONTRACEP2,GESTRISCO,INSC_PN,SEQ_AIH5,CBOR,CNAER,VINCPREV,GESTOR_COD,GESTOR_TP,GESTOR_CPF,GESTOR_DT,CNES,CNPJ_MANT,INFEHOSP,CID_ASSO,CID_MORTE,COMPLEX,FINANC,FAEC_TP,REGCT,RACA_COR,ETNIA,SEQUENCIA,REMESSA,AUD_JUST,SIS_JUST,VAL_SH_FED,VAL_SP_FED,VAL_SH_GES,VAL_SP_GES,VAL_UCI,MARCA_UCI,DIAGSEC1,DIAGSEC2,DIAGSEC3,DIAGSEC4,DIAGSEC5,DIAGSEC6,DIAGSEC7,DIAGSEC8,DIAGSEC9,TPDISEC1,TPDISEC2,TPDISEC3,TPDISEC4,TPDISEC5,TPDISEC6,TPDISEC7,TPDISEC8,TPDISEC9,dh_ingestao_bronze,fonte_origem
431720,2022,05,01,95815668000101,4322100682779,1,98796250,431720,19620602,1,0,0,0,0,00,0,0,0,0,0,1,0407040102,0407040102,400.55,146.96,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,547.51,0.00,104.68,20220128,20220129,K409,0000,12,00,3999,1,0,0,431720,4,59,1,0,010,null,01,0,null,0,0,0,null,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,null,2254611,null,null,0000,0000,02,06,null,0000,01,0000,53,HM43172001N202205.DTS,null,null,0.00,0.00,0.00,0.00,0.00,00,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0,2026-09-25T23:50:27.155Z,DATASUS_SIH_KAGGLE_RS
431720,2022,05,01,95815668000101,4322100682790,1,98787562,431720,19600710,1,0,0,0,0,00,0,0,0,0,1,1,0407040102,0407040102,404.55,146.96,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,551.51,0.00,105.45,20220124,20220125,K409,0000,12,00,3999,1,0,0,431720,4,61,1,0,010,null,01,0,null,0,0,0,null,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,null,2254611,null,null,0000,0000,02,06,null,0000,01,0000,54,HM43172001N202205.DTS,null,null,0.00,0.00,0.00,0.00,0.00,00,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0,2026-09-25T23:50:27.155Z,DATASUS_SIH_KAGGLE_RS
431720,2022,05,01,95815668000101,4322100683252,1,98794076,431720,19981231,1,0,0,0,0,00,0,0,0,0,0,1,0408050160,0408050160,2242.73,332.03,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2574.76,0.00,492.30,20220428,20220429,M235,0000,12,00,3999,1,0,0,431720,4,23,1,0,010,null,01,0,null,0,0,0,null,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,null,2254611,null,null,0000,0000,02,06,null,0000,01,0000,58,HM43172001N202205.DTS,null,null,0.00,0.00,0.00,0.00,0.00,00,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0,2026-09-25T23:50:27.155Z,DATASUS_SIH_KAGGLE_RS
431720,2022,05,03,95815668000101,4322102213077,1,98918000,430220,19490315,3,0,0,0,0,00,0,0,0,0,2,2,0304080020,0304080020,961.79,146.21,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1108.00,0.00,211.85,20220426,20220428,C180,0000,12,00,3999,1,0,0,431720,4,73,2,0,010,null,02,0,null,0,0,0,null,00,00,1,000000000000,000,000000,000,0,00000,0,000000000000000,null,2254611,null,null,0000,0000,03,06,null,0000,01,0000,599,HM43172001N202205.DTS,null,null,0.00,0.00,0.00,0.00,0.00,00,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0,2026-09-25T23:50:27.155Z,DATASUS_SIH_KAGGLE_RS
431720,2022,05,03,95815668000101,4322102257165,1,98280000,431390,19730220,1,0,0,0,4,75,0,0,0,0,0,1,0303140151,0303140151,3306.12,453.35,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,3759.47,2400.00,718.82,20220316,20220321,J180,0000,51,00,3999,1,0,0,431720,4,49,5,0,010,null,02,0,null,0,0,0,null,00,00,1,000000000000,000,000000,000,0,00000,1,000048457990063,null,2254611,null,null,0000,0000,02,06,null,0000,01,0000,600,HM43172001N202205.DTS,null,null,0.00,0.00,0.00,0.00,0.00,00,null,null,null,null,null,null,null,null,null,0,0,0,0,0,0,0,0,0,2026-09-25T23:50:27.155Z,DATASUS_SIH_KAGGLE_RS
431720,2022,05,03,95815668000101,432